# StudyBuddy: University Study Assistant Chatbot

**Domain:** university study planning and programming coursework support.

This notebook implements a functional, interactive chatbot for the CM2015 Programming with Data midterm project. The chatbot uses external JSON intents, regex-based pattern recognition, modular functions, input preprocessing, short-term memory, dynamic response templates, random response selection, sentiment scoring, keyword extraction, and test cases.

The main interactive function is `run_chatbot()`. It exits only when the user types `exit` or `quit`.

## 1. Design summary

StudyBuddy is a rule-based chatbot. The data lives in `intents.json`; the notebook loads that file and builds dictionaries for pattern matching and response generation.

The implementation follows this pipeline:

1. Load intents from JSON.
2. Compile each regex pattern with case-insensitive matching.
3. Preprocess user input by tokenising, removing punctuation, removing stop words, and stemming.
4. Match a regex pattern to an intent.
5. Update short-term memory from named regex groups such as `name`, `course`, `color`, and `deadline_days`.
6. Generate a response by randomly choosing a template and filling placeholders from memory and NLP features.
7. Keep asking for input until the user types `exit` or `quit`.

In [6]:
from __future__ import annotations

import json
import random
import re
from collections import Counter
from pathlib import Path
from string import Formatter
from typing import Any, Dict, Iterable, List, Optional, Tuple

try:
    from nltk.stem import PorterStemmer
except Exception:
    PorterStemmer = None

INTENTS_PATH = Path("intents.json")
EXIT_WORDS = {"exit", "quit"}

# A stop-word list keeps the notebook runnable without internet access
STOP_WORDS = {
    "a", "an", "and", "are", "as", "at", "be", "but", "by", "can", "could", "do", "for",
    "from", "how", "i", "if", "in", "is", "it", "me", "my", "of", "on", "or", "our", "please",
    "should", "so", "the", "this", "to", "was", "we", "what", "when", "where", "who", "why", "with",
    "you", "your", "am", "about", "that", "have", "has", "had", "help", "need", "want"
}

POSITIVE_WORDS = {
    "good", "great", "calm", "confident", "ready", "happy", "clear", "motivated", "helpful", "thanks"
}
NEGATIVE_WORDS = {
    "bad", "stress", "stressed", "overwhelm", "overwhelmed", "anxious", "panic", "worried",
    "tired", "unmotivated", "confused", "behind", "cannot", "can't"
}

stemmer = PorterStemmer() if PorterStemmer is not None else None

## 2. Loading chatbot data from JSON

The JSON file contains intent tags, regex patterns, and response templates. Loading the data externally makes the chatbot easier to maintain because new intents can be added without rewriting the main loop.

In [7]:
def load_intents(path: Path | str = INTENTS_PATH) -> Dict[str, Any]:
    """Load chatbot intents from a JSON file and perform basic validation."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Could not find {path}. Keep intents.json beside this notebook.")

    with path.open("r", encoding="utf-8") as file:
        data = json.load(file)

    if "intents" not in data or not isinstance(data["intents"], list):
        raise ValueError("The intents JSON must contain an 'intents' list.")

    required_keys = {"tag", "patterns", "responses"}
    for index, intent in enumerate(data["intents"]):
        missing = required_keys - intent.keys()
        if missing:
            raise ValueError(f"Intent at index {index} is missing keys: {missing}")
        if not isinstance(intent["patterns"], list) or not isinstance(intent["responses"], list):
            raise TypeError("Each intent must store patterns and responses as lists.")

    return data


def build_chatbot_resources(data: Dict[str, Any]) -> Dict[str, Any]:
    """Build dictionaries used by the chatbot from loaded JSON data."""
    intent2responses: Dict[str, List[str]] = {}
    intent2description: Dict[str, str] = {}
    pattern2intent: Dict[re.Pattern[str], str] = {}

    for intent in data["intents"]:
        tag = intent["tag"]
        intent2responses[tag] = intent["responses"]
        intent2description[tag] = intent.get("description", "")

        for pattern in intent["patterns"]:
            compiled = re.compile(pattern, flags=re.IGNORECASE)
            pattern2intent[compiled] = tag

    return {
        "metadata": data.get("metadata", {}),
        "intent2responses": intent2responses,
        "intent2description": intent2description,
        "pattern2intent": pattern2intent,
    }


intent_data = load_intents(INTENTS_PATH)
resources = build_chatbot_resources(intent_data)

print(f"Loaded {len(resources['intent2responses'])} intents from {INTENTS_PATH}.")
print(f"Compiled {len(resources['pattern2intent'])} regex patterns.")

Loaded 16 intents from intents.json.
Compiled 34 regex patterns.


## 3. Preprocessing and basic NLP

The preprocessing functions split text into tokens, remove non-essential punctuation, remove common stop words, and stem terms to their root form. These processed tokens support sentiment scoring, keyword extraction, and fallback matching.

In [8]:
def tokenize(text: str) -> List[str]:
    """Tokenise text into lowercase word-like units using regular expressions."""
    return re.findall(r"[A-Za-z0-9']+", text.lower())


def stem_token(token: str) -> str:
    """Stem one token using NLTK if available, otherwise use a small suffix-based fallback."""
    if stemmer is not None:
        return stemmer.stem(token)

    for suffix in ("ing", "ed", "ly", "s"):
        if token.endswith(suffix) and len(token) > len(suffix) + 2:
            return token[: -len(suffix)]
    return token


def preprocess(text: str) -> Dict[str, List[str]]:
    """Return raw, filtered, and stemmed tokens for a user message."""
    raw_tokens = tokenize(text)
    filtered_tokens = [token for token in raw_tokens if token not in STOP_WORDS]
    stemmed_tokens = [stem_token(token) for token in filtered_tokens]
    return {
        "raw_tokens": raw_tokens,
        "filtered_tokens": filtered_tokens,
        "stemmed_tokens": stemmed_tokens,
    }


example_preprocess = preprocess("Please help me plan my CM2015 project, tests, and deadlines!")
example_preprocess

{'raw_tokens': ['please',
  'help',
  'me',
  'plan',
  'my',
  'cm2015',
  'project',
  'tests',
  'and',
  'deadlines'],
 'filtered_tokens': ['plan', 'cm2015', 'project', 'tests', 'deadlines'],
 'stemmed_tokens': ['plan', 'cm2015', 'project', 'test', 'deadlin']}

## 4. Regex pattern matching

Each pattern from `intents.json` is compiled with `re.IGNORECASE`. The `find_intent` function searches the raw user text so that regex constructs such as named groups, optional words, lookaheads, `\d`, `\w`, `*`, `+`, and `?` can be used directly.

In [9]:
def find_intent(user_text: str, pattern2intent: Dict[re.Pattern[str], str]) -> Tuple[str, Optional[re.Match[str]]]:
    """Return the first matching intent tag and regex match object."""
    for pattern, tag in pattern2intent.items():
        match = pattern.search(user_text)
        if match:
            return tag, match
    return "fallback", None


for sample in [
    "My name is Maya",
    "Can you plan my CM2015 project deadline in 10 days?",
    "I feel anxious about revision",
    "Show my memory",
]:
    tag, match = find_intent(sample, resources["pattern2intent"])
    print(f"{sample!r} -> {tag}; groups={match.groupdict() if match else {}}")

'My name is Maya' -> set_name; groups={'name': 'Maya'}
'Can you plan my CM2015 project deadline in 10 days?' -> deadline_help; groups={'deadline_days': '10'}
'I feel anxious about revision' -> motivation_stress; groups={}
'Show my memory' -> memory_summary; groups={}


## 5. Memory, sentiment, keyword extraction, and response generation

The chatbot stores memory in a normal Python dictionary. This demonstrates dynamic string substitution: JSON responses contain placeholders such as `{name}`, `{course}`, `{color}`, `{deadline}`, `{keywords}`, and `{sentiment_label}` which are filled at runtime.

In [10]:
def normalise_name(name: str) -> str:
    """Clean and title-case a captured name."""
    return name.strip(" .,!?").replace("’", "'").title()


def normalise_course(course: str) -> str:
    """Clean course/module strings while preserving CM-style course codes."""
    cleaned = re.sub(r"\s+", " ", course.strip(" .,!?")).strip()
    compact = cleaned.replace(" ", "")
    if re.fullmatch(r"[A-Za-z]{2}\d{4}", compact):
        return compact.upper()
    if cleaned.lower() == "programming with data":
        return "CM2015 Programming with Data"
    return cleaned.title()


def update_memory_from_match(memory: Dict[str, str], intent: str, match: Optional[re.Match[str]]) -> None:
    """Update memory using named regex groups captured during intent recognition."""
    if match is None:
        return

    groups = {key: value for key, value in match.groupdict().items() if value}

    if "name" in groups:
        memory["name"] = normalise_name(groups["name"])
    if "course" in groups:
        memory["course"] = normalise_course(groups["course"])
    if "color" in groups:
        memory["color"] = groups["color"].strip(" .,!?").lower()
    if "deadline_days" in groups:
        memory["deadline"] = f"in {groups['deadline_days']} days"
    if "topic" in groups:
        memory["topic"] = re.sub(r"\s+", " ", groups["topic"].lower()).strip()


def analyse_sentiment(tokens: Iterable[str]) -> Dict[str, Any]:
    """Return a simple lexicon-based sentiment label and score."""
    token_set = set(tokens)
    positive = len(token_set & {stem_token(word) for word in POSITIVE_WORDS})
    negative = len(token_set & {stem_token(word) for word in NEGATIVE_WORDS})
    score = positive - negative

    if score > 0:
        label = "positive"
    elif score < 0:
        label = "negative"
    else:
        label = "neutral"

    return {"label": label, "score": score, "positive_hits": positive, "negative_hits": negative}


def extract_keywords(processed_tokens: List[str], limit: int = 5) -> List[str]:
    """Return the most frequent processed tokens as lightweight keyword extraction."""
    if not processed_tokens:
        return []
    counts = Counter(processed_tokens)
    return [token for token, _ in counts.most_common(limit)]


class SafeFormatDict(dict):
    """Leave unknown placeholders readable instead of raising KeyError."""
    def __missing__(self, key: str) -> str:
        return "{" + key + "}"


def make_context(memory: Dict[str, str], nlp: Dict[str, Any]) -> Dict[str, str]:
    """Build values used to fill JSON response templates."""
    name = memory.get("name", "there")
    course = memory.get("course", "your course")
    color = memory.get("color", "not set")
    deadline = memory.get("deadline", "not set")
    topic = memory.get("topic", "that topic")
    keywords = ", ".join(nlp.get("keywords", [])) or "no strong keywords"

    return {
        "name": name,
        "name_phrase": "" if name == "there" else f" {name}",
        "course": course,
        "color": color,
        "deadline": deadline,
        "deadline_phrase": "" if deadline == "not set" else f" {deadline}",
        "topic": topic,
        "keywords": keywords,
        "sentiment_label": nlp.get("sentiment", {}).get("label", "neutral"),
        "sentiment_score": str(nlp.get("sentiment", {}).get("score", 0)),
    }


def generate_response(intent: str, resources: Dict[str, Any], memory: Dict[str, str], nlp: Dict[str, Any], rng: random.Random | None = None) -> str:
    """Choose and format a response template for the detected intent."""
    rng = rng or random
    responses = resources["intent2responses"].get(intent) or resources["intent2responses"]["fallback"]
    template = rng.choice(responses)
    context = SafeFormatDict(make_context(memory, nlp))
    return template.format_map(context)

## 6. Main chatbot logic

The `chatbot_response` function is separated from the `run_chatbot` loop so that it can be tested automatically. The loop itself terminates only when the user types `exit` or `quit`.

In [11]:
def chatbot_response(
    user_text: str,
    resources: Dict[str, Any],
    memory: Dict[str, str] | None = None,
    rng: random.Random | None = None,
) -> Dict[str, Any]:
    """Process one user message and return the response plus debug information."""
    if memory is None:
        memory = {}

    tokens = preprocess(user_text)
    sentiment = analyse_sentiment(tokens["stemmed_tokens"])
    keywords = extract_keywords(tokens["stemmed_tokens"])
    nlp = {"tokens": tokens, "sentiment": sentiment, "keywords": keywords}

    intent, match = find_intent(user_text, resources["pattern2intent"])

    # Fallback enhancement: a clearly negative message should receive the wellbeing/planning response.
    if intent == "fallback" and sentiment["label"] == "negative":
        intent = "motivation_stress"

    update_memory_from_match(memory, intent, match)
    response = generate_response(intent, resources, memory, nlp, rng=rng)

    return {
        "intent": intent,
        "response": response,
        "memory": dict(memory),
        "nlp": nlp,
        "match_groups": match.groupdict() if match else {},
    }


def run_chatbot(path: Path | str = INTENTS_PATH) -> None:
    """Run the interactive chatbot until the user types 'exit' or 'quit'."""
    data = load_intents(path)
    local_resources = build_chatbot_resources(data)
    memory: Dict[str, str] = {}
    rng = random.Random()

    title = local_resources["metadata"].get("chatbot_title", "StudyBuddy")
    print(f"{title}: Hello. Type 'exit' or 'quit' to end the chat.")

    while True:
        user_text = input("You: ").strip()
        if user_text.lower() in EXIT_WORDS:
            print(f"{title}: Goodbye. Your session has ended.")
            break

        result = chatbot_response(user_text, local_resources, memory, rng=rng)
        print(f"{title}: {result['response']}")

# To try the interactive bot in Jupyter, run this in a new cell:
# run_chatbot()

## 7. Demonstration transcript

The following non-interactive demo uses the same `chatbot_response` function as the main loop. A fixed random seed keeps the displayed transcript reproducible.

In [12]:
demo_messages = [
    "Hello",
    "My name is Maya",
    "I am studying CM2015",
    "Can you plan my CM2015 project deadline in 10 days?",
    "I feel anxious and overwhelmed about the workload",
    "What keywords are in this sentence about regex tests and JSON responses?",
    "Show my memory",
    "Thanks"
]

demo_memory: Dict[str, str] = {}
demo_rng = random.Random(42)

for message in demo_messages:
    result = chatbot_response(message, resources, demo_memory, rng=demo_rng)
    print(f"You: {message}")
    print(f"StudyBuddy [{result['intent']}]: {result['response']}")
    print()

You: Hello
StudyBuddy [greeting]: Hello. I am StudyBuddy. I can help with planning, revision, Python concepts, deadlines, and test cases.

You: My name is Maya
StudyBuddy [set_name]: Nice to meet you, Maya. I will remember your name while this chat is running.

You: I am studying CM2015
StudyBuddy [set_course]: Course memory updated to CM2015. Ask me for a project plan, revision strategy, or test-case ideas.

You: Can you plan my CM2015 project deadline in 10 days?
StudyBuddy [deadline_help]: For a deadline in 10 days, use a three-pass plan: make the minimum working version, test it, then improve the report and comments.

You: I feel anxious and overwhelmed about the workload
StudyBuddy [motivation_stress]: I am detecting a negative tone. Shrink the task: choose the next 10-minute action, such as fixing one regex or writing one test.

You: What keywords are in this sentence about regex tests and JSON responses?
StudyBuddy [keyword_request]: The strongest keywords I found are: keyword, 

## 8. Test cases

These tests verify the key behaviours required by the coursework: regex intent recognition, JSON-driven responses, memory substitution, preprocessing, fallback behaviour, and robustness when responses are randomly selected.

In [13]:
def assert_no_unfilled_placeholders(text: str) -> None:
    """Fail if a response still contains an unresolved template placeholder."""
    unresolved = [field_name for _, field_name, _, _ in Formatter().parse(text) if field_name]
    assert not unresolved, f"Unresolved placeholders found in response: {unresolved}"


def run_tests() -> None:
    """Run a compact test suite for the chatbot."""
    test_memory: Dict[str, str] = {}
    test_rng = random.Random(7)

    # Test 1: named regex group updates memory and response uses the stored name.
    result = chatbot_response("My name is Maya", resources, test_memory, rng=test_rng)
    assert result["intent"] == "set_name"
    assert test_memory["name"] == "Maya"
    assert_no_unfilled_placeholders(result["response"])

    # Test 2: flexible regex recognises a deadline request and captures a number with \d{1,3}.
    result = chatbot_response("Can you plan my CM2015 project deadline in 10 days?", resources, test_memory, rng=test_rng)
    assert result["intent"] == "deadline_help"
    assert test_memory["deadline"] == "in 10 days"
    assert_no_unfilled_placeholders(result["response"])

    # Test 3: stress-related input triggers the wellbeing/planning intent and negative sentiment.
    result = chatbot_response("I feel anxious and overwhelmed about the workload", resources, test_memory, rng=test_rng)
    assert result["intent"] == "motivation_stress"
    assert result["nlp"]["sentiment"]["label"] == "negative"
    assert_no_unfilled_placeholders(result["response"])

    # Test 4: keyword extraction uses tokenisation, stop-word removal, and stemming.
    result = chatbot_response("Extract keywords from chatbot architecture modules", resources, test_memory, rng=test_rng)
    assert result["intent"] == "keyword_request"
    assert "chatbot" in result["nlp"]["keywords"]
    assert_no_unfilled_placeholders(result["response"])

    # Test 5: an unknown input is handled safely by fallback.
    result = chatbot_response("purple elephant moon sandwich", resources, test_memory, rng=test_rng)
    assert result["intent"] == "fallback"
    assert "not sure" in result["response"].lower() or "did not" in result["response"].lower()
    assert_no_unfilled_placeholders(result["response"])

    print("All chatbot tests passed.")


run_tests()

All chatbot tests passed.


## 9. Process reflection

The chatbot was developed iteratively.

- **Week 1:** selected the domain and wrote the first set of intents. Early review showed that hard-coded responses would be difficult to maintain, so the data was moved into `intents.json`.
- **Week 2:** added modular functions for loading JSON, compiling regex, matching intents, and generating responses. Feedback from testing showed that exact string matching was too rigid, so regex patterns were expanded with optional words, case-insensitive matching, `\d`, grouping, and lookaheads.
- **Week 3:** added preprocessing, stemming, keyword extraction, sentiment scoring, and memory. This improved robustness because the chatbot could still extract useful information even when punctuation and wording varied.
- **Week 4:** added test cases, a reproducible demo transcript, and a report. The tests focus on behaviour rather than exact random response text, which makes them more robust.

Potential future improvements include a larger intent dataset, a richer sentiment model, persistent memory saved to a file, and a graphical user interface.